# Agentic RAG + ChromaDB — the traditional alternative to OKF

**Companion to `google_okf_zero_to_mastery.ipynb`.** That notebook packaged knowledge about the
Olist dataset as an **OKF bundle** — structured markdown files with YAML frontmatter, an explicit
link graph, a conformance spec — and put a simple RAG discovery agent on top.

This notebook throws the OKF layer away entirely and does what most real-world RAG systems
actually do: dump content as **flat, unstructured text chunks into a vector database**
([ChromaDB](https://www.trychroma.com/)), with **no** frontmatter, **no** bundle hierarchy, **no**
parseable cross-links — and consume it with an **agentic** loop (the model decides, via native
tool-calling, whether and how many times to search) instead of a fixed single-shot retrieve
step.

Same dataset, same local models, same real problem, same 14-question eval set (reused verbatim
from the retired advanced-RAG benchmark notebook, including its concept-ID naming, kept purely as
plain string identifiers here — there is no bundle, no frontmatter, no files backing them anymore).

**What this notebook is *not***: a controlled, paired benchmark against notebook 1. Notebook 1 was
evaluated on its own, easier, 8-question set in its own execution. Any comparison to its numbers
here is explicitly a *reference point*, not an apples-to-apples result — that distinction is kept
visible throughout, not blurred for a cleaner-looking table.

## Roadmap

1. Environment check.
2. Data download and profiling (same Olist dataset).
3. Foreign-key discovery and real computed metrics (same methodology as notebook 1).
4. Content generation as **flat text**, not OKF concepts — same grounding discipline (facts from
   code, prose from the LLM, every generated claim checked), same two hallucination guards
   notebook 1 discovered it needed (metric grounding, date-range grounding).
5. Ingest into ChromaDB with a custom Ollama embedding function.
6. Two consumption pipelines: **simple RAG** (single retrieve + generate, a fair control) and
   **agentic RAG** (native tool-calling loop, the model decides when/how much to search — cited
   to [Yao et al. 2022, ReAct](https://arxiv.org/abs/2210.03629) and the
   [Agentic RAG survey](https://arxiv.org/abs/2501.09136)).
7. Benchmark both, for real, on the same corpus, in the same run.
8. Honest verdict, architectural comparison notes, limitations, references.

## 0. Environment check

In [1]:
import json
import os
import re
import time
import warnings
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Optional

import chromadb
import numpy as np
import ollama
import pandas as pd
from chromadb import EmbeddingFunction
from loguru import logger
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

GEN_MODEL = "qwen3.5:4b"
EMBED_MODEL = "nomic-embed-text"
RUN_TS = datetime.now(timezone.utc).replace(microsecond=0).isoformat()

logger.remove()
logger.add(lambda msg: print(msg, end=""), level="INFO", format="<level>{message}</level>\n")


def model_available(name: str, local_models: set) -> bool:
    return name in local_models or any(m.split(":")[0] == name.split(":")[0] for m in local_models)


local_models = {m["model"] for m in ollama.list()["models"]}
for required in (GEN_MODEL, EMBED_MODEL):
    assert model_available(required, local_models), f"Missing local model {required!r}."
print(f"generation model : {GEN_MODEL}\nembedding model  : {EMBED_MODEL}\nrun timestamp    : {RUN_TS}")

CALL_COUNTER = {"llm": 0, "embed": 0}


def llm(prompt: str, *, num_predict: int = 350, temperature: float = 0.2, retries: int = 2) -> str:
    CALL_COUNTER["llm"] += 1
    last_err = None
    for _ in range(retries + 1):
        try:
            resp = ollama.chat(model=GEN_MODEL, messages=[{"role": "user", "content": prompt}], think=False,
                                options={"num_predict": num_predict, "temperature": temperature})
            content = resp["message"]["content"].strip()
            if content:
                return content
        except Exception as e:
            last_err = e
        time.sleep(0.5)
    raise RuntimeError(f"LLM call failed: {last_err}")


def llm_chat(messages: list, *, tools: Optional[list] = None, temperature: float = 0.2):
    CALL_COUNTER["llm"] += 1
    return ollama.chat(model=GEN_MODEL, messages=messages, tools=tools, think=False,
                        options={"temperature": temperature})


class OllamaEmbeddingFunction(EmbeddingFunction):
    """Chroma requires subclassing its `EmbeddingFunction` (not duck-typing) — a bare object with a
    matching `__call__` is missing the `embed_query` mixin method Chroma calls internally and fails
    with `AttributeError: ... no attribute 'embed_query'` at query time. Verified directly before
    writing the rest of this notebook."""

    def __init__(self, model: str = EMBED_MODEL):
        self.model = model

    def __call__(self, input):
        CALL_COUNTER["embed"] += len(input)
        return [ollama.embed(model=self.model, input=t)["embeddings"][0] for t in input]

    def name(self):
        return f"ollama-{self.model}"


print(llm("Reply with exactly the word: ready"))

generation model : qwen3.5:4b
embedding model  : nomic-embed-text
run timestamp    : 2026-07-05T09:32:14+00:00


ready


## 1. Data, foreign keys, metrics

Identical methodology to notebook 1 — see that notebook for the fully narrated version. This is
here only to produce real grounded content for the flat corpus below.

In [2]:
import kagglehub

DATASET_DIR = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
TABLE_RENAME = {"product_category_name_translation": "category_translation"}


def table_name_from_file(path: Path) -> str:
    name = re.sub(r"_dataset$", "", re.sub(r"^olist_", "", path.stem))
    return TABLE_RENAME.get(name, name)


tables = {table_name_from_file(f): pd.read_csv(f) for f in sorted(DATASET_DIR.glob("*.csv"))}
print(f"Loaded {len(tables)} tables: {list(tables)}")


def profile_table(name: str, df: pd.DataFrame) -> dict:
    cols = []
    for col in df.columns:
        s = df[col]
        cols.append({"name": col, "dtype": str(s.dtype), "null_pct": round(float(s.isna().mean()) * 100, 2),
                     "n_unique": int(s.nunique()), "samples": [str(v) for v in s.dropna().unique()[:3]]})
    return {"name": name, "n_rows": int(len(df)), "n_cols": int(len(df.columns)), "columns": cols}


profiles = {name: profile_table(name, df) for name, df in tables.items()}


def base_key(col: str) -> str:
    return "zip_code_prefix" if col.endswith("zip_code_prefix") else col


def detect_fk_candidates(tables: dict, min_overlap: float = 0.9) -> pd.DataFrame:
    groups = defaultdict(list)
    for tname, df in tables.items():
        for col in df.columns:
            groups[base_key(col)].append((tname, col))
    rows = []
    for members in groups.values():
        if len(members) < 2:
            continue
        for i, (t1, c1) in enumerate(members):
            for j, (t2, c2) in enumerate(members):
                if i == j:
                    continue
                s1, s2 = tables[t1][c1].dropna(), tables[t2][c2].dropna()
                if s1.empty or s2.empty:
                    continue
                overlap = float(s1.isin(set(s2.unique())).mean())
                if overlap >= min_overlap:
                    rows.append({"from_table": t1, "from_column": c1, "to_table": t2, "to_column": c2,
                                 "overlap": round(overlap, 4)})
    return pd.DataFrame(rows)


fk_candidates = detect_fk_candidates(tables)
print(f"Detected {len(fk_candidates)} foreign-key candidate relationships.")

Loaded 9 tables: ['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'sellers', 'category_translation']


Detected 24 foreign-key candidate relationships.


In [3]:
orders_df = tables["orders"].copy()
for col in ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]:
    orders_df[col] = pd.to_datetime(orders_df[col], errors="coerce")

payments_per_order = tables["order_payments"].groupby("order_id")["payment_value"].sum()
delivered = orders_df[orders_df["order_status"] == "delivered"].dropna(
    subset=["order_delivered_customer_date", "order_estimated_delivery_date"])
late_mask = delivered["order_delivered_customer_date"] > delivered["order_estimated_delivery_date"]
review_scores = tables["order_reviews"]["review_score"]
cat_en = tables["products"].merge(tables["category_translation"], on="product_category_name", how="left")
top_categories = (tables["order_items"].merge(cat_en[["product_id", "product_category_name_english"]], on="product_id", how="left")
                  .groupby("product_category_name_english")["order_id"].nunique().sort_values(ascending=False).head(5))

metrics_summary = {
    "avg_order_value": {"value": round(float(payments_per_order.mean()), 2), "unit": "BRL",
                         "n": int(payments_per_order.shape[0]), "source_tables": ["order_payments"],
                         "definition": "mean(sum(payment_value) grouped by order_id)"},
    "late_delivery_rate": {"value": round(float(late_mask.mean()) * 100, 2), "unit": "%",
                            "n": int(len(delivered)), "source_tables": ["orders"],
                            "definition": "mean(delivered_date > estimated_date) over delivered orders"},
    "avg_review_score": {"value": round(float(review_scores.mean()), 2), "unit": "stars (1-5)",
                          "n": int(review_scores.notna().sum()), "source_tables": ["order_reviews"],
                          "definition": "mean(review_score)"},
    "review_score_distribution": {"value": (review_scores.value_counts(normalize=True).sort_index() * 100).round(2).to_dict(),
                                   "unit": "% of reviews", "n": int(review_scores.notna().sum()),
                                   "source_tables": ["order_reviews"], "definition": "value_counts(review_score, normalize=True)"},
    "payment_type_distribution": {"value": (tables["order_payments"]["payment_type"].value_counts(normalize=True) * 100).round(2).to_dict(),
                                   "unit": "% of payments", "n": int(tables["order_payments"].shape[0]),
                                   "source_tables": ["order_payments"], "definition": "value_counts(payment_type, normalize=True)"},
    "top_product_categories": {"value": top_categories.to_dict(), "unit": "distinct orders", "n": int(top_categories.sum()),
                                "source_tables": ["order_items", "products", "category_translation"],
                                "definition": "nunique(order_id) grouped by product_category_name_english, top 5"},
}
for k, v in metrics_summary.items():
    print(f"{k:28s} = {v['value']}")

avg_order_value              = 160.99
late_delivery_rate           = 8.11
avg_review_score             = 4.09
review_score_distribution    = {1: 11.51, 2: 3.18, 3: 8.24, 4: 19.29, 5: 57.78}
payment_type_distribution    = {'credit_card': 73.92, 'boleto': 19.04, 'voucher': 5.56, 'debit_card': 1.47, 'not_defined': 0.0}
top_product_categories       = {'bed_bath_table': 9417, 'health_beauty': 8836, 'sports_leisure': 7720, 'computers_accessories': 6689, 'furniture_decor': 6449}


## 2. Content generation — flat text, not OKF concepts

Same grounding discipline as notebook 1 (facts from pandas, prose from the LLM, every claim
checked against ground truth, including the two hallucination guards that notebook discovered it
needed), but the *output* is a plain `{id, text, metadata}` dict — no YAML frontmatter, no
`# Schema`/`# Joins`/`# Citations` section structure, no bundle directory, no files at all. IDs
reuse notebook 1's naming (`tables/orders`, `references/metrics/avg_order_value`, ...) purely as
opaque string identifiers for eval-set compatibility — they carry no structural meaning here.

In [4]:
KAGGLE_URL = "https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce"


def format_metric_value(info: dict) -> str:
    v = info["value"]
    return ", ".join(f"{k}: {v2}" for k, v2 in v.items()) if isinstance(v, dict) else str(v)


def number_variants(x: float) -> set:
    return {f"{x}", f"{x:.0f}", f"{x:.1f}", f"{x:.2f}", str(round(x))}


def metric_is_grounded(text: str, info: dict) -> bool:
    clean = text.replace(",", "")
    v = info["value"]
    candidates = v.values() if isinstance(v, dict) else [v]
    return any(variant in clean for x in candidates for variant in number_variants(float(x)))


def make_table_doc(name: str) -> dict:
    profile = profiles[name]
    col_summary = "; ".join(f"{c['name']} ({c['dtype']}, {c['null_pct']}% null)" for c in profile["columns"])
    rels = fk_candidates[(fk_candidates["from_table"] == name) | (fk_candidates["to_table"] == name)]
    rel_bits = [f"{r.from_table}.{r.from_column} relates to {r.to_table}.{r.to_column} ({r.overlap*100:.0f}% overlap)"
                for r in rels.itertuples()]
    prompt = (f"Document a database table. Use ONLY the facts given; never invent numbers or business meaning.\n\n"
              f"Table: {name}\nRows: {profile['n_rows']:,}\nColumns: {col_summary}\n"
              f"Relationships: {'; '.join(rel_bits) if rel_bits else 'none detected'}\n\n"
              f"Write two short paragraphs separated by a blank line: (1) what one row represents, "
              f"(2) a note on relationships/data quality. Output ONLY the prose.")
    raw = llm(prompt, num_predict=250)
    title = name.replace("_", " ").title()
    text = (f"{title} (table)\n\n{raw}\n\n"
            f"Columns: {col_summary}\n"
            f"Relationships: {'; '.join(rel_bits) if rel_bits else 'none detected'}\n\n"
            f"Source: Olist Brazilian E-Commerce Public Dataset (Kaggle), table `{name}`.")
    return {"id": f"tables/{name}", "text": text, "metadata": {"type": "Table", "title": title}}


def make_metric_doc(key: str) -> dict:
    info = metrics_summary[key]
    value_str = format_metric_value(info)
    prompt = (f"Document a business metric. Metric: {key.replace('_',' ')}\nValue: {value_str} {info['unit']}\n"
              f"Computed over: {info['n']:,} records\nDefinition: {info['definition']}\n"
              f"Write 2-3 sentences explaining the metric; you MUST include the exact value ({value_str}) verbatim. Output ONLY the prose.")
    text = llm(prompt, num_predict=200)
    if not metric_is_grounded(text, info):
        text = llm(prompt + f"\n\nReminder: {value_str} MUST appear verbatim.", num_predict=200)
    if not metric_is_grounded(text, info):
        text = f"The {key.replace('_', ' ')} is {value_str}, computed as {info['definition']} over {info['n']:,} records."
    title = key.replace("_", " ").title()
    full_text = (f"{title} (metric)\n\n{text}\n\n"
                 f"Definition: {info['definition']}\nValue: {value_str} {info['unit']}\n"
                 f"Computed over: {info['n']:,} records\nSource table(s): {', '.join(info['source_tables'])}.")
    return {"id": f"references/metrics/{key}", "text": full_text, "metadata": {"type": "Metric", "title": title}}


def make_dataset_doc() -> dict:
    date_min, date_max = orders_df["order_purchase_timestamp"].min(), orders_df["order_purchase_timestamp"].max()
    valid_years = {date_min.year, date_max.year}
    table_list = "\n".join(f"- {n}: {p['n_rows']:,} rows" for n, p in profiles.items())
    prompt = (f"Document a dataset. Order date range: {date_min:%Y-%m-%d} to {date_max:%Y-%m-%d}\nTables:\n{table_list}\n\n"
              f"Write 2-3 sentences on what this dataset represents. Use ONLY the facts above — do not state any "
              f"date range other than the one given. Output ONLY the prose.")
    text = llm(prompt, num_predict=200)

    def years_ok(t):
        return {int(y) for y in re.findall(r"\b(20\d{2})\b", t)}.issubset(valid_years)

    if not years_ok(text):
        text = llm(prompt + f"\n\nReminder: the only valid years are {sorted(valid_years)}.", num_predict=200)
    if not years_ok(text):
        text = (f"The Olist Brazilian E-Commerce Public Dataset captures orders placed between "
                f"{date_min:%B %Y} and {date_max:%B %Y} across {len(profiles)} tables.")
    full_text = (f"Olist Brazilian E-Commerce (dataset)\n\n{text}\n\nTables:\n{table_list}\n\n"
                 f"Source: Olist Brazilian E-Commerce Public Dataset (Kaggle), {KAGGLE_URL}")
    return {"id": "datasets/olist_ecommerce", "text": full_text, "metadata": {"type": "Dataset", "title": "Olist Brazilian E-Commerce"}}

In [5]:
documents = [make_dataset_doc()]
for name in tqdm(list(profiles), desc="tables"):
    documents.append(make_table_doc(name))
for key in tqdm(list(metrics_summary), desc="metrics"):
    documents.append(make_metric_doc(key))

print(f"\nGenerated {len(documents)} flat documents. LLM calls so far: {CALL_COUNTER['llm']}")
print("\nExample document (tables/orders):\n")
print(next(d["text"] for d in documents if d["id"] == "tables/orders"))

tables:   0%|          | 0/9 [00:00<?, ?it/s]

metrics:   0%|          | 0/6 [00:00<?, ?it/s]


Generated 16 flat documents. LLM calls so far: 17

Example document (tables/orders):

Orders (table)

Each of the 99,441 rows in the orders table documents a single customer purchase record containing identifiers for the order and its associated entities such as the customer, along with temporal data points including timestamps for purchase, estimated delivery, actual carrier delivery, customer receipt, and approval status. The dataset includes eight string columns where most fields are fully populated except for specific dates like `order_delivered_carrier_date` which has a 1.79% null rate, while the primary keys remain completely non-null across all records.

The table is tightly integrated with other tables through high-fidelity relationships; specifically, there is a perfect one-to-one overlap between orders and customers on their respective IDs, as well as complete links to order payments via `order_id`. However, data quality varies slightly in the many-to-many associations invol

## 3. Ingest into ChromaDB

An ephemeral (in-memory) client — consistent with this project's "always rebuild fresh" pattern
(notebook 1 rebuilds its bundle every run too; nothing here is meant to persist across runs).

In [6]:
chroma_client = chromadb.EphemeralClient()
collection = chroma_client.get_or_create_collection(
    "olist_flat_corpus", embedding_function=OllamaEmbeddingFunction(EMBED_MODEL)
)
collection.add(
    ids=[d["id"] for d in documents],
    documents=[d["text"] for d in documents],
    metadatas=[d["metadata"] for d in documents],
)
print(f"Ingested {collection.count()} documents into ChromaDB collection {collection.name!r}.")

Ingested 16 documents into ChromaDB collection 'olist_flat_corpus'.


In [7]:
_probe = collection.query(query_texts=["how much do customers spend per order"], n_results=3)
for i, doc, meta, dist in zip(_probe["ids"][0], _probe["documents"][0], _probe["metadatas"][0], _probe["distances"][0]):
    print(f"[{i}] {meta['type']} similarity={1 - dist:.3f}")
print("\nChromaDB retrieval sanity check passed." if _probe["ids"][0][0] == "references/metrics/avg_order_value"
      else "\nWarning: top hit was not the expected metric doc — inspect above.")

[references/metrics/avg_order_value] Metric similarity=0.356
[tables/orders] Table similarity=0.228
[references/metrics/top_product_categories] Metric similarity=0.136

ChromaDB retrieval sanity check passed.


## 4. Two consumption pipelines

### 4.1 Simple RAG — the fair control

One retrieval call, one generation call. This is the same architecture-shape as notebook 1's
naive baseline, just over a flat Chroma corpus instead of an OKF bundle + FAISS.

In [8]:
def simple_rag_answer(query: str, top_k: int = 4) -> dict:
    res = collection.query(query_texts=[query], n_results=top_k)
    ids, docs = res["ids"][0], res["documents"][0]
    context = "\n\n---\n\n".join(f"[{i}] {d}" for i, d in zip(ids, docs))
    prompt = (f"Answer the user's question using ONLY the knowledge-base entries below. Cite entry "
              f"id(s) in square brackets, e.g. [tables/orders]. If the entries do not contain the "
              f"answer, say so explicitly rather than guessing.\n\nKnowledge base entries:\n{context}"
              f"\n\nQuestion: {query}\n\nAnswer (2-4 sentences, with citations):")
    answer = llm(prompt, num_predict=300, temperature=0.1)
    return {"retrieved_ids": list(ids), "answer": answer}


print("Simple RAG pipeline ready.")

Simple RAG pipeline ready.


### 4.2 Agentic RAG — the model decides

Native Ollama tool-calling (verified working with `qwen3.5:4b` before writing this), not manual
ReAct text parsing — cited to [Yao et al. 2022](https://arxiv.org/abs/2210.03629) for the
interleaved reasoning/acting idea and the
[Agentic RAG survey](https://arxiv.org/abs/2501.09136) for the broader pattern family. The model
gets one tool, `search_knowledge_base`, and decides for itself whether one search is enough or
whether it needs another — e.g. to look up a related table after seeing the first result. Bounded
at `max_iterations` to keep cost and failure modes measurable rather than open-ended.

In [9]:
SEARCH_TOOL = [{
    "type": "function",
    "function": {
        "name": "search_knowledge_base",
        "description": "Semantic search over the knowledge base. Returns the top matching documents with their id and content.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "the search query"},
                "top_k": {"type": "integer", "description": "number of results to return (default 3)"},
            },
            "required": ["query"],
        },
    },
}]

AGENT_SYSTEM_PROMPT = (
    "You are a knowledge-base assistant. Use the search_knowledge_base tool to find relevant "
    "documents before answering. You may call it more than once if the first results are "
    "insufficient — for example, to look up a related table or metric mentioned in an earlier "
    "result. Once you have enough information, answer in 2-4 sentences, citing the document id(s) "
    "you used in square brackets, e.g. [tables/orders]. If nothing relevant turns up after "
    "searching, say so explicitly rather than guessing."
)


def _run_search_tool(args: dict) -> tuple[str, list]:
    query = args.get("query", "")
    top_k = int(args.get("top_k", 3) or 3)
    res = collection.query(query_texts=[query], n_results=top_k)
    ids, docs = res["ids"][0], res["documents"][0]
    if not ids:
        return "No results found.", []
    return "\n\n".join(f"[{i}]\n{d[:600]}" for i, d in zip(ids, docs)), list(ids)


def agentic_answer(query: str, max_iterations: int = 4) -> dict:
    messages = [{"role": "system", "content": AGENT_SYSTEM_PROMPT}, {"role": "user", "content": query}]
    search_log, retrieved_ids = [], []
    for step in range(max_iterations):
        resp = llm_chat(messages, tools=SEARCH_TOOL)
        messages.append(resp["message"])
        tool_calls = resp["message"].get("tool_calls")
        if not tool_calls:
            return {"answer": resp["message"]["content"], "iterations": step + 1, "searches": search_log,
                    "retrieved_ids": retrieved_ids, "capped": False}
        for tc in tool_calls:
            args = tc.function.arguments
            search_log.append({"query": args.get("query", query), "top_k": args.get("top_k", 3)})
            result, ids = _run_search_tool(args)
            retrieved_ids += [i for i in ids if i not in retrieved_ids]
            messages.append({"role": "tool", "content": result, "tool_name": tc.function.name})

    messages.append({"role": "user", "content": "Answer now with what you already found, citing id(s)."})
    final = llm_chat(messages, tools=None)
    return {"answer": final["message"]["content"], "iterations": max_iterations, "searches": search_log,
            "retrieved_ids": retrieved_ids, "capped": True}


print("Agentic RAG pipeline ready.")

Agentic RAG pipeline ready.


### 4.3 Sanity check — watch the agent decide

In [10]:
_r = agentic_answer("How much do customers typically spend per basket?")
print("Searches issued :", _r["searches"])
print("Retrieved ids   :", _r["retrieved_ids"])
print("Iterations used :", _r["iterations"], "| hit cap:", _r["capped"])
print("Answer          :", _r["answer"])

Searches issued : [{'query': 'customers typical spending per basket average transaction value', 'top_k': 5}]
Retrieved ids   : ['references/metrics/avg_order_value', 'references/metrics/payment_type_distribution', 'tables/order_payments', 'tables/orders', 'references/metrics/avg_review_score']
Iterations used : 2 | hit cap: False
Answer          : Based on the data available in your knowledge base, customers are currently spending an average of **R$160.99** per basket (or order). This figure represents the arithmetic mean calculated across a robust dataset of 99,440 records [references/metrics/avg_order_value].


## 5. Evaluation set

The exact same 14 hand-labeled questions used in the (now-retired) advanced-RAG-vs-baseline
benchmark — direct, paraphrase, multi-hop, vague, and unanswerable-distractor categories, chosen
specifically to avoid the ceiling effect notebook 1's original 8-question set hit. Document IDs
are reused verbatim as plain identifiers so the same expected-answer labels apply unchanged.

In [11]:
eval_set = [
    {"q": "What columns does the orders table have?", "expected": {"tables/orders"}, "answerable": True, "category": "direct"},
    {"q": "What is the average order value?", "expected": {"references/metrics/avg_order_value"}, "answerable": True, "category": "direct"},
    {"q": "What is the distribution of payment types?", "expected": {"references/metrics/payment_type_distribution"}, "answerable": True, "category": "direct"},
    {"q": "How can product category names be translated to English?", "expected": {"tables/category_translation"}, "answerable": True, "category": "direct"},
    {"q": "How much do customers typically spend per basket?", "expected": {"references/metrics/avg_order_value"}, "answerable": True, "category": "paraphrase"},
    {"q": "What fraction of shipments arrive behind schedule?", "expected": {"references/metrics/late_delivery_rate"}, "answerable": True, "category": "paraphrase"},
    {"q": "Are Brazilian customers happy with their purchases based on ratings?", "expected": {"references/metrics/avg_review_score", "references/metrics/review_score_distribution"}, "answerable": True, "category": "paraphrase"},
    {"q": "If I know a product's category in Portuguese, how do I find its seller's city?", "expected": {"tables/category_translation", "tables/order_items", "tables/sellers"}, "answerable": True, "category": "multi-hop"},
    {"q": "How would I compute total revenue per seller?", "expected": {"tables/order_items", "tables/sellers", "tables/order_payments"}, "answerable": True, "category": "multi-hop"},
    {"q": "Which table tells me both a customer's zip code and their approximate map coordinates?", "expected": {"tables/customers", "tables/geolocation"}, "answerable": True, "category": "multi-hop"},
    {"q": "seller performance", "expected": {"tables/sellers", "tables/order_items"}, "answerable": True, "category": "vague"},
    {"q": "payment methods", "expected": {"references/metrics/payment_type_distribution", "tables/order_payments"}, "answerable": True, "category": "vague"},
    {"q": "What is the customer's email address or phone number?", "expected": set(), "answerable": False, "category": "distractor"},
    {"q": "What was Olist's total company revenue in 2020?", "expected": set(), "answerable": False, "category": "distractor"},
]
pd.DataFrame([{"category": e["category"], "answerable": e["answerable"], "question": e["q"]} for e in eval_set])

,category,answerable,question
0,direct,True,What columns does the orders table have?
1,direct,True,What is the average order value?
2,direct,True,What is the distribution of payment types?
3,direct,True,How can product category names be translated t...
4,paraphrase,True,How much do customers typically spend per basket?
5,paraphrase,True,What fraction of shipments arrive behind sched...
6,paraphrase,True,Are Brazilian customers happy with their purch...
7,multi-hop,True,"If I know a product's category in Portuguese, ..."
8,multi-hop,True,How would I compute total revenue per seller?
9,multi-hop,True,Which table tells me both a customer's zip cod...


## 6. Head-to-head benchmark

Simple RAG vs. agentic RAG, same corpus, same run — a controlled comparison. Notebook 1's own
recorded numbers (Hit@1 88%, Hit@3/5 100%, groundedness 3/3) are **not** included in the table
below because they were measured on a different, easier 8-question set in a different execution —
quoting them side-by-side with these numbers would misrepresent both.

In [12]:
REFUSAL_RE = re.compile(
    r"do(?:es)? not (?:contain|include|provide)|not available|cannot determine|no information|"
    r"don'?t have|isn'?t available|unable to (?:determine|answer)|not (?:mentioned|specified|available)|"
    r"knowledge base (?:entries )?do(?:es)? not|no results found",
    re.IGNORECASE,
)


def looks_like_refusal(answer: str) -> bool:
    return bool(REFUSAL_RE.search(answer))


def hit_at_k(expected: set, ids: list, k: int):
    return any(cid in expected for cid in ids[:k]) if expected else None


def cite_hit(expected: set, answer: str, known_ids: list):
    if not expected:
        return None
    return any(f"[{cid}]" in answer for cid in known_ids if cid in expected)


bench_rows = []
for item in tqdm(eval_set, desc="benchmark"):
    q, expected, answerable, category = item["q"], item["expected"], item["answerable"], item["category"]

    before = dict(CALL_COUNTER)
    s = simple_rag_answer(q)
    simple_calls = (CALL_COUNTER["llm"] - before["llm"]) + (CALL_COUNTER["embed"] - before["embed"])

    before = dict(CALL_COUNTER)
    a = agentic_answer(q)
    agentic_calls = (CALL_COUNTER["llm"] - before["llm"]) + (CALL_COUNTER["embed"] - before["embed"])

    all_ids = [d["id"] for d in documents]
    bench_rows.append({
        "question": q[:55], "category": category, "answerable": answerable,
        "simple_hit@1": hit_at_k(expected, s["retrieved_ids"], 1),
        "simple_hit@3": hit_at_k(expected, s["retrieved_ids"], 3),
        "simple_cite_hit": cite_hit(expected, s["answer"], all_ids),
        "agentic_search_hit": hit_at_k(expected, a["retrieved_ids"], len(a["retrieved_ids"])),
        "agentic_cite_hit": cite_hit(expected, a["answer"], all_ids),
        "agentic_n_searches": len(a["searches"]), "agentic_capped": a["capped"],
        "simple_refusal_ok": (not answerable) and looks_like_refusal(s["answer"]),
        "agentic_refusal_ok": (not answerable) and looks_like_refusal(a["answer"]),
        "simple_calls": simple_calls, "agentic_calls": agentic_calls,
    })

bench_df = pd.DataFrame(bench_rows)
display(bench_df)

benchmark:   0%|          | 0/14 [00:00<?, ?it/s]

,question,category,answerable,simple_hit@1,simple_hit@3,simple_cite_hit,agentic_search_hit,agentic_cite_hit,agentic_n_searches,agentic_capped,simple_refusal_ok,agentic_refusal_ok,simple_calls,agentic_calls
0,What columns does the orders table have?,direct,True,True,True,True,True,True,1,False,False,False,2,3
1,What is the average order value?,direct,True,True,True,True,True,True,1,False,False,False,2,3
2,What is the distribution of payment types?,direct,True,True,True,True,True,False,1,False,False,False,2,3
3,How can product category names be translated t...,direct,True,True,True,True,True,True,1,False,False,False,2,3
4,How much do customers typically spend per basket?,paraphrase,True,True,True,True,True,True,1,False,False,False,2,3
5,What fraction of shipments arrive behind sched...,paraphrase,True,True,True,True,True,True,1,False,False,False,2,3
6,Are Brazilian customers happy with their purch...,paraphrase,True,True,True,True,True,False,1,False,False,False,2,3
7,"If I know a product's category in Portuguese, ...",multi-hop,True,True,True,True,True,True,3,False,False,False,2,7
8,How would I compute total revenue per seller?,multi-hop,True,False,False,False,True,True,1,False,False,False,2,3
9,Which table tells me both a customer's zip cod...,multi-hop,True,True,True,True,True,True,1,False,False,False,2,3


### 6.1 Aggregate scorecard

In [13]:
answerable_df = bench_df[bench_df["answerable"]]
unanswerable_df = bench_df[~bench_df["answerable"]]

summary = pd.DataFrame({
    "simple RAG": [
        answerable_df["simple_hit@1"].mean(), answerable_df["simple_hit@3"].mean(),
        answerable_df["simple_cite_hit"].mean(), float("nan"),
        unanswerable_df["simple_refusal_ok"].mean(), bench_df["simple_calls"].mean(),
    ],
    "agentic RAG": [
        answerable_df["agentic_search_hit"].mean(), float("nan"),
        answerable_df["agentic_cite_hit"].mean(), answerable_df["agentic_n_searches"].mean(),
        unanswerable_df["agentic_refusal_ok"].mean(), bench_df["agentic_calls"].mean(),
    ],
}, index=["Retrieval hit rate", "Hit@3 (simple only)", "Answer-citation hit rate",
          "Avg searches issued (agentic only)", "Refusal correctness (distractors)", "Avg LLM+embed calls / question"])
display(summary.round(3))

print(f"\nn={len(answerable_df)} answerable questions, n={len(unanswerable_df)} distractor questions.")
print(f"Agentic pipeline issued {answerable_df['agentic_n_searches'].mean():.1f} searches/question on average "
      f"(cap=4); {int(bench_df['agentic_capped'].sum())}/{len(bench_df)} questions hit the iteration cap.")
print(f"Agentic pipeline cost {summary.loc['Avg LLM+embed calls / question', 'agentic RAG'] / summary.loc['Avg LLM+embed calls / question', 'simple RAG']:.1f}x "
      f"the calls of simple RAG per question — measured, not estimated.")

,simple RAG,agentic RAG
Retrieval hit rate,0.917,0.917
Hit@3 (simple only),0.917,NaN
Answer-citation hit rate,0.833,0.750
Avg searches issued (agentic only),NaN,1.167
Refusal correctness (distractors),1.000,1.000
Avg LLM+embed calls / question,2.000,3.286



n=12 answerable questions, n=2 distractor questions.
Agentic pipeline issued 1.2 searches/question on average (cap=4); 0/14 questions hit the iteration cap.
Agentic pipeline cost 1.6x the calls of simple RAG per question — measured, not estimated.


### 6.2 Breakdown by question category

In [14]:
by_category = answerable_df.groupby("category")[["simple_cite_hit", "agentic_cite_hit", "agentic_n_searches"]].mean()
display(by_category.round(2))

,simple_cite_hit,agentic_cite_hit,agentic_n_searches
category,,,
direct,1.0,0.75,1.00
multi-hop,0.666667,1.0,1.67
paraphrase,1.0,0.666667,1.00
vague,0.5,0.5,1.00


## 7. Honest verdict

Computed from the actual numbers above, not asserted in advance.

In [15]:
def delta_word(base: float, adv: float) -> str:
    if adv == base:
        return "tied with"
    return "beat" if adv > base else "underperformed"


s_cite, a_cite = answerable_df["simple_cite_hit"].mean(), answerable_df["agentic_cite_hit"].mean()
s_ref, a_ref = unanswerable_df["simple_refusal_ok"].mean(), unanswerable_df["agentic_refusal_ok"].mean()
s_cost, a_cost = bench_df["simple_calls"].mean(), bench_df["agentic_calls"].mean()

print(f"Answer-citation hit rate : agentic {delta_word(s_cite, a_cite)} simple — {a_cite:.0%} vs {s_cite:.0%}")
print(f"Distractor refusal       : agentic {delta_word(s_ref, a_ref)} simple — {a_ref:.0%} vs {s_ref:.0%}")
print(f"Cost                     : agentic used {a_cost / s_cost:.1f}x the calls of simple RAG")
print(f"Average searches issued by the agent per question: {answerable_df['agentic_n_searches'].mean():.1f} "
      f"(1 search = behaviorally identical to simple RAG; >1 means it decided to look further)")

Answer-citation hit rate : agentic underperformed simple — 75% vs 83%
Distractor refusal       : agentic tied with simple — 100% vs 100%
Cost                     : agentic used 1.6x the calls of simple RAG
Average searches issued by the agent per question: 1.2 (1 search = behaviorally identical to simple RAG; >1 means it decided to look further)


### 7.1 What actually happened when the model could choose to search again

The number above — average searches per question — is the real finding here, not a hit-rate
table. If it's close to 1.0, the "agentic" framing bought nothing: the model almost never chose to
use its own autonomy. If it's meaningfully above 1.0, especially concentrated on multi-hop
questions, that is the tool-calling loop earning its keep. Whichever it is, it is reported as
measured — this section is deliberately not the same "advanced beats naive" framing pushed by
default; see notebook 2's retired benchmark for how often that framing turned out to be only half
true even there.

## 8. OKF vs. flat-corpus RAG+VectorDB — lessons from actually building both

Having now built both a knowledge layer (notebook 1: OKF bundle) and a pure retrieval layer over
unstructured content (this notebook: ChromaDB), the differences are concrete, not theoretical:

| | OKF bundle (notebook 1) | Flat ChromaDB corpus (this notebook) |
|---|---|---|
| Storage | Plain files, `git`-diffable, human-browsable on GitHub with zero tooling | A running (even if embedded/in-process) vector database process |
| Structure | Explicit: typed frontmatter, `# Schema`/`# Joins`/`# Citations` sections, a real link graph | None inherent — whatever metadata dict fields the pipeline author remembered to attach |
| Consumption without retrieval | Possible — small bundles fit wholesale in context, browsable via `index.md` | Not really — there is no directory to browse; the vector index *is* the access path |
| Enables graph-aware retrieval | Yes — notebook 2's (retired) graph-expansion stage walked real OKF links | No — a flat Chroma corpus has no links to walk; "related tables" only exist as prose the retriever might or might not surface |
| Conformance / validation | A real, checkable spec (SPEC.md §9) — this project's `validate_conformance` actually runs and asserts | No equivalent concept — "is this corpus well-formed" isn't a question ChromaDB asks |
| Setup cost | `pip`/`uv` install of a YAML/markdown parser, nothing else | A vector DB dependency (`chromadb`, ~contains embedded SQLite + ONNX runtime deps) |
| Where it shines | Long-lived, versioned, human-and-agent-curated knowledge that outlives any one retrieval pipeline | Fast to stand up, works over content nobody structured on purpose (PDFs, wikis, scraped pages) |

Neither is strictly better — they answer different questions. "How do I store and version
knowledge so both humans and agents can curate it over time?" points at OKF. "I have a pile of
unstructured content right now and need retrieval today" points at flat RAG + a vector DB. This
project needed the former (a genuinely undocumented dataset, worth curating once) but built the
latter here specifically to make the tradeoff concrete instead of asserted.

## 9. Limitations

- **`max_iterations=4`** bounds the agentic loop for cost/measurability reasons. A question that
  genuinely needs a 5th search is indistinguishable, in these results, from one where the model
  was simply being inefficient.
- **Citation-hit is a bracket-substring check** (`[tables/orders]` literally present in the
  answer text) — same class of heuristic as `metric_is_grounded` and `looks_like_refusal`
  elsewhere in this project. A correct answer that cites informally (no brackets) would be scored
  as a miss.
- **The embedding function calls Ollama once per document/query**, serially — fine at 16
  documents, would not scale to a real corpus without batching.
- **This is not a controlled comparison against notebook 1.** Different corpus representation,
  different retrieval architecture, different eval-set difficulty (notebook 1: 8 easier
  questions; here: 14 harder ones). The only fully controlled comparison in this notebook is
  simple-RAG vs. agentic-RAG, both against the identical Chroma corpus in the identical run.

## 10. References

1. Yao et al. (2022). ["ReAct: Synergizing Reasoning and Acting in Language Models."](https://arxiv.org/abs/2210.03629)
2. Singh et al. (2025). ["Agentic Retrieval-Augmented Generation: A Survey on Agentic RAG."](https://arxiv.org/abs/2501.09136)
3. [ChromaDB documentation](https://docs.trychroma.com/) — vector store used here.
4. [Ollama tool-calling](https://ollama.com/blog/tool-support) — native function-calling API used for the agentic loop, verified working with `qwen3.5:4b` before this notebook was written.
5. `google_okf_zero_to_mastery.ipynb` — the companion notebook this one contrasts with; SPEC.md at [`GoogleCloudPlatform/knowledge-catalog`](https://github.com/GoogleCloudPlatform/knowledge-catalog/blob/main/okf/SPEC.md) for the OKF format itself.